## Can multi-listing (“professional”) hosts be distinguished from single-listing hosts, and what operating differences show up in the data?

# Data pre-processing

**Implemented**
- Handle missing values for bedrooms and baths
- Property type consolidation
- Host listings count -> single/multi listing flag

**Not implemented**
- Minimum-stay discretisation
- Distance-from-CBD discretisation
- Add number of listings nearby to this property
---
We choose to implement the first three tasks listed.
- The single/multi listing flag is necessary as our research question involves predicting which hosts fall into which category.
- We chose to also implement property-type consolidation as we believed this may be correlated with host types, and reducing data scarcity by having fewer, more populated buckets will allow us to draw more meaningful relationships.
- Out of the remaining possible tasks, we felt that filling the gaps in bathroom and bedroom count would be the most informative when trying to derive potential relationships with host type

**Host listings count**  
This task is simple - if the Airbnb-reported number of listings by the host is greater than 1, flag them as multi-listing; otherwise flag them as single-listing. This applies to every single row (since every row has `host_listings_count`)

**Property type consolidation**  
This could have been done algorithmically by attempting to process each of the unique property types using some global ruleset, however there is a lot of variation in the types present which would be difficult to create a full ruleset for (eg. 'kezhan', 'castle', 'train'). Instead, we created a few simple rules (eg. remove anything relevant to the room type, so we only deal with the property type; remove 'entire' if present before the property type). Then we manually sorted each of the remaining types into a few discrete buckets that we felt represented the broad themes present. These were:
- Apartment
- House
- Homestay
- Hotel
- Vehicle
- Novel
- Other

This affected 100% of rows as well, and reduced the number of categories from 82 to 7.

**Missing values for bedrooms and bathrooms**  
A combination of approaches was used here to impute as many missing values as possible.

For bedrooms, if the `bedrooms` count was not available, we first checked the `room_type`. If this was either "Shared room" or "Private room", we set `bedrooms_calculated` to 1; this was based off our observation that listings under this room type were all single-rooms. If there was a different room type, we attempted to find the number of rooms from the description using a regular expression, and if this failed we would set the `bedrooms_calculated` to NaN. We left 459 rows as NaN, or 1.78%, which is an improvement from the original 18.19% that was unavailable.

For bathrooms, if `bathrooms` was missing we would use a regular expression on `bathrooms_text`. An alternative approach we could have tried was to try find the value in `description`, but we found that every single property with bathrooms listed in the description also had it in `bathrooms_text`, and the only other properties didn't have their bathrooms listed anywhere. For these, we just left `bathrooms_calculated` as NaN. This improved the number of unknown rows from 31.71% to 0.09%.

In [217]:
import pandas as pd
import re

df = pd.read_csv('listings.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25728 entries, 0 to 25727
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            25728 non-null  int64  
 1   listing_url                                   25728 non-null  str    
 2   scrape_id                                     25728 non-null  int64  
 3   last_scraped                                  25728 non-null  str    
 4   source                                        25728 non-null  str    
 5   name                                          25728 non-null  str    
 6   description                                   25276 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   25728 non-null  str    
 9   host_id                                       25728 non-null  int64  
 1

In [218]:
## Single vs multi-listing count
df['host_type'] = df['calculated_host_listings_count'].apply(
    lambda x: 'single-listing' if x==1 else 'multi-listing'
)

In [219]:
## Consolidate property_types into fewer categories:
# Apartment
# House
# Homestay
# Hotel
# Vehicle
# Novel
# Other

def consolidate_property_type(raw_string):
    cleaned = raw_string.lower()
    cleaned = re.sub(r'.*?\broom\b( in)? ?', '', cleaned)  # remove '___ room in...'
    cleaned = re.sub(r'^entire\s?', '', cleaned)  # remove 'Entire ...'

    consolidated_types = {
        'apartment': ['rental unit', 'serviced apartment', 'condo', 'home/apt'],  # checked home/apt
        'house': ['cabin', 'tiny home', 'earthen home', 'home',
                  'house', 'cottage', 'villa', 'chalet', 'vacation home', 'nature lodge',
                  'bungalow', 'casa particular', 'townhouse'],
        'homestay': ['bed and breakfast', 'floor', 'guest suite', 'guesthouse'],  # checked floor
        'hotel': ['boutique hotel', 'aparthotel', 'hostel', 'hotel', 'resort', 'holiday park'],
        'vehicle': ['bus', 'camper/rv', 'boat', 'train'],
        'novel': ['yurt', 'tipi', 'treehouse', 'barn', 'castle', 'tent', 'farm stay',
                  'dome', 'religious building', 'hut'],
        'other/unknown': ['loft', '', 'place', 'tower', 'minsu', 'kezhan']  # since different meanings
    }

    for key, values in consolidated_types.items():
        if cleaned in values:
            return key

    return 'other/unknown'  # fallback which shouldn't trigger with current dataset

df['consolidated_property_type'] = df['property_type'].apply(consolidate_property_type)

In [220]:
## Missing values for bedrooms
# If room type is private room or shared room, set bedrooms to 1
# Else try scraping using regex
# Otherwise leave nan

def extract_bedrooms(row):
    if not pd.isna(row['bedrooms']):
        return row['bedrooms']
    elif row['room_type'] in ["Private room", "Shared room"]:
        return 1.0
    else:
        result = re.search(r'(\d+(\.\d+)?)[ -]?(?i:bedroom|bedrooms|bdr|br)\b', str(row['description']))
        if result is not None:
            return float(result.group(1))
        else:
            return float('nan')

df['bedrooms_calculated'] = df.apply(extract_bedrooms, axis=1)


# df.loc[pd.isna(df['bedrooms']), ['bedrooms', 'room_type', 'description', 'bedrooms_calculated']]
num_unknown = len(df.loc[df['bedrooms_calculated'].isnull()])
num_unknown_before = len(df.loc[df['bedrooms'].isnull()])
num_total = len(df)
print(f"% unknown before: {round(num_unknown_before / num_total * 100, 2)}%")
print(f"% unknown now:    {round(num_unknown / num_total * 100, 2)}%")

% unknown before: 18.19%
% unknown now:    1.78%


In [221]:
## Missing values for bathrooms
# Use bathrooms column
# Else extract from bathrooms_text
# ** If fails, try extracting from description  <---- tried this, none of them work so dropped it
# Else leave nan

def extract_bathrooms(row):
    if not pd.isna(row['bathrooms']):
        return row['bathrooms']
    elif not pd.isna(row['bathrooms_text']):
        num = re.search(r'(\d+(\.\d+)?) (shared |private )?baths?', str(row['bathrooms_text']))
        halves = re.search(r'(?i:half-bath)', str(row['bathrooms_text']))
        if num is not None:
            return float(num.group(1))
        elif halves is not None:
            return 0.5
    else: 
        return float('nan')
    

df['bathrooms_calculated'] = df.apply(extract_bathrooms, axis=1)


# df.loc[df['bathrooms_calculated'].isna(), ['description', 'bathrooms', 'bathrooms_text', 'bathrooms_calculated']]
num_unknown = len(df.loc[df['bathrooms_calculated'].isnull()])
num_unknown_before = len(df.loc[df['bathrooms'].isnull()])
num_total = len(df)
print(f"% unknown before: {round(num_unknown_before / num_total * 100, 2)}%")
print(f"% unknown now:    {round(num_unknown / num_total * 100, 2)}%")

len(df.loc[df['bathrooms_calculated'].isna(), ['description', 'bathrooms', 'bathrooms_text']])
len(df)

% unknown before: 31.71%
% unknown now:    0.09%


25728